In [16]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from imblearn.under_sampling import TomekLinks


In [17]:
TW_1000= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1000/Twitter-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_1000.columns = columns

### RandomUnderSampler for Random Forest

In [18]:
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [19]:
ratios = {
    "1:10": 0.1,
    "1:15": 0.067,
    "1:20": 0.05,
    "1:25": 0.04,
    "1:30": 0.033
}
results = {}

for name, ratio in ratios.items():
    print(f"\n===== Undersampling {name} =====")

    rus = RandomUnderSampler(sampling_strategy=ratio, random_state=42)
    X_res, y_res = rus.fit_resample(X_train, y_train)

    print("After:", y_res.value_counts())

    rf = RandomForestClassifier(random_state=42)

    param_grid = {
        'n_estimators': [100],
        'max_depth': [20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    grid = GridSearchCV( rf, param_grid, scoring='f1', cv=3, n_jobs=-1)
    grid.fit(X_res, y_res)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    results[name] = f1

    print("Best params:", grid.best_params_)
    print("F1:", f1)

print("\n===== FINAL RESULTS =====")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


===== Undersampling 1:10 =====
After: label
0.0    9420
1.0     942
Name: count, dtype: int64
Best params: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
F1: 0.5

===== Undersampling 1:15 =====
After: label
0.0    14059
1.0      942
Name: count, dtype: int64
Best params: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
F1: 0.5824345146379045

===== Undersampling 1:20 =====
After: label
0.0    18840
1.0      942
Name: count, dtype: int64
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
F1: 0.6239460370994941

===== Undersampling 1:25 =====
After: label
0.0    23550
1.0      942
Name: count, dtype: int64
Best params: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
F1: 0.6520947176684881

===== Undersampling 1:30 =====
After: label
0.0    28545
1.0      942
Name: count, dtype: int64
Best params: {'max_depth': None, 'min_samp

### RandomUnderSampler for XGBoost

In [20]:

ratios = {
    "1:10": 0.1,
    "1:15": 0.067,
    "1:20": 0.05,
    "1:25": 0.04,
    "1:30": 0.033
}

results = {}

for name, ratio in ratios.items():
    print(f"\n===== Undersampling {name} =====")

    rus = RandomUnderSampler(sampling_strategy=ratio, random_state=42)
    X_res, y_res = rus.fit_resample(X_train, y_train)

    print("After:", y_res.value_counts())

    xgb = XGBClassifier(random_state=42,eval_metric='logloss')

    param_grid = {
        'n_estimators': [100],
        'max_depth': [4, 6],
        'learning_rate': [0.05, 0.1]
    }

    grid = GridSearchCV(xgb,param_grid,scoring='f1',cv=3,n_jobs=-1)
    grid.fit(X_res, y_res)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    results[name] = f1

    print("Best params:", grid.best_params_)
    print("F1:", f1)

print("\n===== FINAL RESULTS =====")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


===== Undersampling 1:10 =====
After: label
0.0    9420
1.0     942
Name: count, dtype: int64
Best params: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100}
F1: 0.4900497512437811

===== Undersampling 1:15 =====
After: label
0.0    14059
1.0      942
Name: count, dtype: int64
Best params: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}
F1: 0.5701624815361891

===== Undersampling 1:20 =====
After: label
0.0    18840
1.0      942
Name: count, dtype: int64
Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100}
F1: 0.6025641025641025

===== Undersampling 1:25 =====
After: label
0.0    23550
1.0      942
Name: count, dtype: int64
Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100}
F1: 0.6339754816112084

===== Undersampling 1:30 =====
After: label
0.0    28545
1.0      942
Name: count, dtype: int64
Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100}
F1: 0.6346863468634686

===== FINAL RESULTS =====
1:10: 0

### Tomek Links for XGBoost

In [21]:
from imblearn.under_sampling import TomekLinks
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report

# --- Tomek Links ---
tl = TomekLinks()
X_train_clean, y_train_clean = tl.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts())
print("After:", y_train_clean.value_counts())

# --- model ---
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

# --- grid ---
param_grid = {
    'n_estimators': [100],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

# --- grid search ---
grid = GridSearchCV(
    xgb,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_clean, y_train_clean)

# --- evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Before: label
0.0    111623
1.0       942
Name: count, dtype: int64
After: label
0.0    111475
1.0       942
Name: count, dtype: int64
Best params: {'colsample_bytree': 1, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100, 'subsample': 0.8}
F1: 0.6713947990543735
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     27907
         1.0       0.76      0.60      0.67       235

    accuracy                           1.00     28142
   macro avg       0.88      0.80      0.83     28142
weighted avg       0.99      1.00      0.99     28142



### Tomek Links  Random Forest

In [22]:
tl = TomekLinks()
X_train_clean, y_train_clean = tl.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts())
print("After:", y_train_clean.value_counts())

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100],
    'max_depth': [20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    rf,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_clean, y_train_clean)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Before: label
0.0    111623
1.0       942
Name: count, dtype: int64
After: label
0.0    111475
1.0       942
Name: count, dtype: int64
Best params: {'class_weight': None, 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
F1: 0.6666666666666666
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     27907
         1.0       0.77      0.59      0.67       235

    accuracy                           1.00     28142
   macro avg       0.88      0.79      0.83     28142
weighted avg       0.99      1.00      0.99     28142



### ADASYN XGBoost

In [23]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

# --- data ---
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']

# --- split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- ADASYN ---
adasyn = ADASYN(sampling_strategy=0.1, random_state=42)
X_res, y_res = adasyn.fit_resample(X_train, y_train)

print("After:", y_res.value_counts())

# --- model ---
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

# --- grid ---
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1]
}

# --- grid search ---
grid = GridSearchCV(
    xgb,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_res, y_res)

# --- evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

After: label
0.0    111623
1.0     11092
Name: count, dtype: int64
Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
F1: 0.6137566137566137
              precision    recall  f1-score   support

         0.0       1.00      0.99      1.00     27907
         1.0       0.52      0.74      0.61       235

    accuracy                           0.99     28142
   macro avg       0.76      0.87      0.80     28142
weighted avg       0.99      0.99      0.99     28142



### ADASYN XGBoost Random Forest

In [24]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import ADASYN
from sklearn.ensemble import RandomForestClassifier

# --- data ---
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']

# --- split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- ADASYN ---
adasyn = ADASYN(sampling_strategy=0.1, random_state=42)
X_res, y_res = adasyn.fit_resample(X_train, y_train)

print("After:", y_res.value_counts())

# --- model ---
rf = RandomForestClassifier(
    random_state=42
)

# --- grid ---
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# --- grid search ---
grid = GridSearchCV(
    rf,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_res, y_res)

# --- evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

After: label
0.0    111623
1.0     11092
Name: count, dtype: int64
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
F1: 0.65625
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     27907
         1.0       0.61      0.71      0.66       235

    accuracy                           0.99     28142
   macro avg       0.80      0.86      0.83     28142
weighted avg       0.99      0.99      0.99     28142



### RUSBOOST 

In [26]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report
from imblearn.ensemble import RUSBoostClassifier

X = TW_1000.drop(columns=['label'])
y = TW_1000['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- model ---
rusboost = RUSBoostClassifier(random_state=42)

# --- grid ---
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2]
}

# --- grid search ---
grid = GridSearchCV(
    rusboost,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

# --- evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best params: {'learning_rate': 0.2, 'n_estimators': 50}
F1: 0.19298245614035087
              precision    recall  f1-score   support

         0.0       1.00      0.94      0.97     27907
         1.0       0.11      0.89      0.19       235

    accuracy                           0.94     28142
   macro avg       0.55      0.91      0.58     28142
weighted avg       0.99      0.94      0.96     28142

